In [74]:
from __future__ import annotations

import numpy as np
from minitorch.tensor.tensor import Tensor

In [75]:
%reload_ext autoreload
%autoreload 3

In [76]:

import numpy as np

from minitorch.tensor.tensor import Tensor
from minitorch.activations.activations import GELU
from minitorch.nn.layers import Linear, Layer
from minitorch.attention.attention import MultiHeadAttention
from minitorch.embendding.embed import EmbeddingLayer


def create_causal_mask(seq_len: int) -> Tensor:
    """
    Create a causal mask (autoregressive mask).
    
    This create the causal mask to make sure that tokens i
    only communicates to token j where j<i.
    Essential for autoregressive GPT models.

    Args:
        seq_len (int): Length of the sequence

    Returns:
        Tensor: Tensor of shape (1, seq_len, seq_len) with:
        - 1.0 for positions that CAN be attended to (lower triangle)
        - 0.0 for positions that CANNOT be attended to (upper triangle)
    """
    mask = np.tril(np.ones(shape=(seq_len, seq_len), dtype= np.float32))
    return Tensor(mask[np.newaxis, :, :])

In [91]:
nin = 3
nlayers = 4

xs = [2.0, 3.0, -1.0, 3.0, -1.0, 0.5, 0.7, 0.8, 0.9],
    # [3.0, -1.0, 0.5],
    # [0.5, 1.0, 1.0],
    # [0.1, 0.2, 0.3],
    # [0.4, 0.5, 0.6],
    # [0.7, 0.8, 0.9]


ys = [1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0,-1.0]

In [98]:
x = Tensor(np.array(xs, dtype=np.float32), requires_grad=True)
y = Tensor(np.array(np.array(ys)), requires_grad=True)

In [116]:
in_features = x.shape[-1]
out_features = 20
nn = Layer(in_features, out_features, num_neurons=3, bias=True)
out = nn(x)

In [117]:
loss = sum([(ygt - ypr) ** 2 for ygt, ypr in zip(y,out)], start=0)
loss.backward()

In [111]:
params = nn.parameters()
w, b = params[0][0], params[0][1]

In [121]:
out[0].reshape(2,10)

Tensor(data=[[1.65731373 1.82520375 0.92896381 0.67780797 1.70502968 1.8531164
  2.29027035 1.54064402 0.76495965 1.3358757 ]
 [2.14602566 1.17059929 1.71658121 1.8228549  1.87398046 1.03757195
  1.35652516 1.07613336 1.22057804 2.40165996]], shape=(2, 10), grad_info= True)

In [129]:
from minitorch.nn.layers import Flatten

Flatten()(input=out[0], shape=(4,5))

Tensor(data=[[1.65731373 1.82520375 0.92896381 0.67780797 1.70502968]
 [1.8531164  2.29027035 1.54064402 0.76495965 1.3358757 ]
 [2.14602566 1.17059929 1.71658121 1.8228549  1.87398046]
 [1.03757195 1.35652516 1.07613336 1.22057804 2.40165996]], shape=(4, 5), grad_info= True)

In [248]:
from minitorch.embendding.embed import Embedding, PositionalEncoding, EmbeddingLayer
from minitorch.tokenization.tokenizer import BPETokenizer, CharTokenizer

In [325]:
corpus = [
    'The City of London serves as the administrative capital of UK',
    'Tottenham Hotspurs is a football club',
    'Am the very best in the World!']
tokenizer = BPETokenizer()
tokenizer.train(corpus)
token_ids = tokenizer.encode('Tottenham Hotspur is the best football club in London!')
# decode = tokenizer.decode(token_ids)
token_ids

[76, 80, 0, 83, 34, 97, 89, 92, 98, 46, 1]

In [319]:
ids = Tensor(np.array(token_ids))
ids = ids.reshape(1,ids.shape[0])
ids

Tensor(data=[[ 83  88  89  40 104  95  98 105  52]], shape=(1, 9), grad_info= False)

In [326]:
tokenizer.tokens_to_ids

{'<UNK>': 0,
 '!<EOT>': 1,
 'a': 2,
 'a<EOT>': 3,
 'b': 4,
 'b<EOT>': 5,
 'c': 6,
 'd': 7,
 'd<EOT>': 8,
 'e': 9,
 'e<EOT>': 10,
 'f': 11,
 'f<EOT>': 12,
 'h': 13,
 'i': 14,
 'k<EOT>': 15,
 'l': 16,
 'l<EOT>': 17,
 'm': 18,
 'm<EOT>': 19,
 'n': 20,
 'n<EOT>': 21,
 'o': 22,
 'p': 23,
 'r': 24,
 's': 25,
 's<EOT>': 26,
 't': 27,
 't<EOT>': 28,
 'u': 29,
 'v': 30,
 'w': 31,
 'y<EOT>': 32,
 'th': 33,
 'the<EOT>': 34,
 'ot': 35,
 'it': 36,
 'of<EOT>': 37,
 'er': 38,
 'am<EOT>': 39,
 'cit': 40,
 'city<EOT>': 41,
 'lo': 42,
 'lon': 43,
 'lond': 44,
 'londo': 45,
 'london<EOT>': 46,
 'ser': 47,
 'serv': 48,
 'serve': 49,
 'serves<EOT>': 50,
 'as<EOT>': 51,
 'ad': 52,
 'adm': 53,
 'admi': 54,
 'admin': 55,
 'admini': 56,
 'adminis': 57,
 'administ': 58,
 'administr': 59,
 'administra': 60,
 'administrat': 61,
 'administrati': 62,
 'administrativ': 63,
 'administrative<EOT>': 64,
 'ca': 65,
 'cap': 66,
 'capit': 67,
 'capita': 68,
 'capital<EOT>': 69,
 'uk<EOT>': 70,
 'tot': 71,
 'tott': 72,
 't

In [312]:
import random

vocab_size = tokenizer.vocab_size
seq_length = ids.shape[-1]
embed_dim = 55
embed = Embedding(vocab_size, embed_dim)
pos_emb = PositionalEncoding(seq_length, 55)

idx = Tensor(np.array([random.randint(0, vocab_size) for _ in range(vocab_size // 2)]))
emb = embed(idx) + pos_emb(ids)

IndexError: index 21 is out of bounds for axis 0 with size 21

In [307]:
emb.shape

(1, 10, 55)

In [308]:
emb = EmbeddingLayer(vocab_size, embed_dim, ids.shape[-1])
embed = emb(ids)
embed

Tensor(data=[[[-0.65698079 -1.03910605  1.36535721 ... -1.62664148  0.64779824
    1.42463398]
  [ 0.45768282 -1.59059138 -1.35830642 ... -1.36494975  0.469852
    1.77532652]
  [-0.95959914  1.02563649  0.26456811 ... -1.88850703 -1.00908607
    1.59759066]
  ...
  [ 2.03245715 -0.00737172 -0.07548454 ...  0.83854278  2.1406037
   -1.39800064]
  [-1.76475869  0.56085359 -0.12810937 ...  0.01143368 -1.86737069
   -0.62822769]
  [ 1.55126867 -1.3547464   1.02527214 ... -1.81193169 -0.38603215
    0.63165327]]], shape=(1, 55, 55), grad_info= True)

In [293]:
import math 

logits = np.exp(embed[0][0].data)
probs = logits / logits.sum()

In [298]:
probs

array([0.00412207, 0.02836495, 0.01094566, 0.00369783, 0.00355269,
       0.01694137, 0.0093081 , 0.03725961, 0.04403704, 0.04773347,
       0.00327491, 0.02694092, 0.03840692, 0.04616933, 0.03015811,
       0.00328251, 0.00376752, 0.00799324, 0.01158476, 0.00871916,
       0.00826459, 0.01749051, 0.02701871, 0.0153169 , 0.01800226,
       0.02430947, 0.00533786, 0.01308424, 0.01221213, 0.01660517,
       0.00569918, 0.02142247, 0.00695426, 0.03915441, 0.01647007,
       0.06507109, 0.00598261, 0.00764401, 0.02508073, 0.00374359,
       0.04017148, 0.00357771, 0.03459064, 0.00440436, 0.00405888,
       0.00862444, 0.01955126, 0.01086001, 0.0051889 , 0.01860284,
       0.00503204, 0.01545404, 0.00449962, 0.03972444, 0.0445349 ])